# Assignment - SDWA Data: Read from S3, Athena, and EDA

1. Read data from a public S3 bucket.
2. Register the data with Athena.
3. Run exploratory data analysis (EDA) in SageMaker.

In [1]:
import boto3
import sagemaker
import pandas as pd
import awswrangler as wr

sess = sagemaker.Session()
bucket = sess.default_bucket()
region = boto3.Session().region_name

print("Default bucket:", bucket)
print("Region:        ", region)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Default bucket: sagemaker-us-east-1-933153382103
Region:         us-east-1


In [7]:
# Set the source path and peek at the data
source_csv = "s3://aai540-group5-public-866792937762-us-east-1-an/sdwa/raw/SDWA_EVENTS_MILESTONES.csv"

df_sample = wr.s3.read_csv(source_csv, nrows=5)
print(df_sample.shape)
df_sample

(5, 10)


,SUBMISSIONYEARQUARTER,PWSID,EVENT_SCHEDULE_ID,EVENT_END_DATE,EVENT_ACTUAL_DATE,EVENT_COMMENTS_TEXT,EVENT_MILESTONE_CODE,EVENT_REASON_CODE,FIRST_REPORTED_DATE,LAST_REPORTED_DATE
0,2026Q1,AL0001223,TTT-AL1813870,09/07/2022,10/06/2022,SUBMIT LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...,RTL1,L1TD,02/23/2023,11/18/2025
1,2026Q1,AL0001566,TTT-AL1816160,10/24/2022,11/19/2022,SUBMIT LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...,RTL1,L1TD,02/23/2023,03/30/2026
2,2026Q1,IN2492893,TTT-IN516540,10/19/2022,11/03/2022,LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...,RTL2,L2TB,03/07/2025,12/17/2025
3,2026Q1,IN2520141,TTT-IN544720,04/06/2023,04/16/2023,LVL2 TTT TC+/EC- WO RPTS 2ND LVL 1-12 MN;0;Sam...,RTL2,L2TB,03/07/2025,02/24/2026
4,2026Q1,IN2860031,TTT-IN598640,08/22/2024,09/15/2024,LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...,RTL2,L2TB,03/07/2025,02/24/2026


In [8]:
# Load the full file
df = wr.s3.read_csv(source_csv)
print("Rows:", df.shape[0], " Cols:", df.shape[1])
print(df.dtypes)
df.head()

Rows: 360370  Cols: 10
SUBMISSIONYEARQUARTER    object
PWSID                    object
EVENT_SCHEDULE_ID        object
EVENT_END_DATE           object
EVENT_ACTUAL_DATE        object
EVENT_COMMENTS_TEXT      object
EVENT_MILESTONE_CODE     object
EVENT_REASON_CODE        object
FIRST_REPORTED_DATE      object
LAST_REPORTED_DATE       object
dtype: object


,SUBMISSIONYEARQUARTER,PWSID,EVENT_SCHEDULE_ID,EVENT_END_DATE,EVENT_ACTUAL_DATE,EVENT_COMMENTS_TEXT,EVENT_MILESTONE_CODE,EVENT_REASON_CODE,FIRST_REPORTED_DATE,LAST_REPORTED_DATE
0,2026Q1,AL0001223,TTT-AL1813870,09/07/2022,10/06/2022,SUBMIT LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...,RTL1,L1TD,02/23/2023,11/18/2025
1,2026Q1,AL0001566,TTT-AL1816160,10/24/2022,11/19/2022,SUBMIT LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...,RTL1,L1TD,02/23/2023,03/30/2026
2,2026Q1,IN2492893,TTT-IN516540,10/19/2022,11/03/2022,LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...,RTL2,L2TB,03/07/2025,12/17/2025
3,2026Q1,IN2520141,TTT-IN544720,04/06/2023,04/16/2023,LVL2 TTT TC+/EC- WO RPTS 2ND LVL 1-12 MN;0;Sam...,RTL2,L2TB,03/07/2025,02/24/2026
4,2026Q1,IN2860031,TTT-IN598640,08/22/2024,09/15/2024,LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...,RTL2,L2TB,03/07/2025,02/24/2026


In [9]:
# Create an Athena database (safe to re-run)
database_name = "sdwa_db"
wr.catalog.create_database(name=database_name, exist_ok=True)
print("Database ready:", database_name)

Database ready: sdwa_db


In [10]:
# Write the data to your private bucket as Parquet and auto-register in Athena
table_name = "events_milestones"
parquet_path = "s3://{}/sdwa/events_milestones/".format(bucket)

wr.s3.to_parquet(
    df=df,
    path=parquet_path,
    dataset=True,
    database=database_name,
    table=table_name,
    mode="overwrite",
)
print("Table registered:", database_name + "." + table_name)

Table registered: sdwa_db.events_milestones


In [11]:
# Verify the Athena table with a SQL query
sql_df = wr.athena.read_sql_query(
    sql="SELECT * FROM {} LIMIT 5".format(table_name),
    database=database_name,
)
sql_df

,submissionyearquarter,pwsid,event_schedule_id,event_end_date,event_actual_date,event_comments_text,event_milestone_code,event_reason_code,first_reported_date,last_reported_date
0,2026Q1,AL0001223,TTT-AL1813870,09/07/2022,10/06/2022,SUBMIT LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...,RTL1,L1TD,02/23/2023,11/18/2025
1,2026Q1,AL0001566,TTT-AL1816160,10/24/2022,11/19/2022,SUBMIT LEVEL 1 TCR ASSESMENT;0;Sample Lab ID:...,RTL1,L1TD,02/23/2023,03/30/2026
2,2026Q1,IN2492893,TTT-IN516540,10/19/2022,11/03/2022,LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...,RTL2,L2TB,03/07/2025,12/17/2025
3,2026Q1,IN2520141,TTT-IN544720,04/06/2023,04/16/2023,LVL2 TTT TC+/EC- WO RPTS 2ND LVL 1-12 MN;0;Sam...,RTL2,L2TB,03/07/2025,02/24/2026
4,2026Q1,IN2860031,TTT-IN598640,08/22/2024,09/15/2024,LVL2 TTT MULTIPLE TC+ 2ND LVL 1-12 MN;0;Sample...,RTL2,L2TB,03/07/2025,02/24/2026


In [12]:
# EDA: shape, dtypes, missing values, summary stats
print("Shape:", df.shape)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Summary statistics:")
df.describe(include="all")

Shape: (360370, 10)

Missing values per column:
SUBMISSIONYEARQUARTER         0
PWSID                         0
EVENT_SCHEDULE_ID             0
EVENT_END_DATE           150232
EVENT_ACTUAL_DATE             0
EVENT_COMMENTS_TEXT      105722
EVENT_MILESTONE_CODE          0
EVENT_REASON_CODE         14819
FIRST_REPORTED_DATE           0
LAST_REPORTED_DATE        29506
dtype: int64

Summary statistics:


,SUBMISSIONYEARQUARTER,PWSID,EVENT_SCHEDULE_ID,EVENT_END_DATE,EVENT_ACTUAL_DATE,EVENT_COMMENTS_TEXT,EVENT_MILESTONE_CODE,EVENT_REASON_CODE,FIRST_REPORTED_DATE,LAST_REPORTED_DATE
count,360370,360370,360370,210138,360370,254648,360370,345551,360370,330864
unique,1,96547,287788,5955,8837,82649,13,15,1758,1370
top,2026Q1,NM3528522,150,10/16/2024,10/16/2024,SUBMIT LEAD SERVICE LINE INVENTORY,SDFF,GW,03/20/2026,03/31/2026
freq,360370,151,3461,8298,44886,33600,107823,78589,8988,26028
